# PPI specificity tutorial — binders that pick their target

This notebook designs minibinders against the **Trk neurotrophin receptors** and scores them for
the following:

- **Specificity** — bind human TrkA, and *not* its close relatives TrkB and TrkC.
- **Multi-specificity** — bind TrkA of human, mouse *and* rabbit

Both use the same machinery: a **pre-oracle transform** that keeps the designed binder, swaps the
target for each structure you name, and refolds. Whether a strong off-target score is a pass or a
fail is entirely up to the threshold you write.

In three steps we will:

- turn the specificity filter on in a **basic inference** run and read the verdict per design;
- repeat it for multi-specificity, where the off-targets are wanted rather than avoided;
- wrap the same run in a **reward-guided search** that spends its compute on the trajectories
  that are working.

> Run this notebook with the repository venv. Sampling is submitted to Slurm; the notebook only
> submits and waits. The AF3 oracle needs the apptainer setup described in the
> [inference guide](../guides/inference.md).

## Before we start: the two mechanisms this rests on

Nothing below needs new Python — it is two knobs from the [metrics guide](../guides/metrics.md), so a
one-paragraph refresher on each before we start.

**`metrics_tags`** is a list of strings you pass at the run level. Every metric, oracle, pass
criterion and filter in the config declares the tags it wants — `required_tags_any`,
`required_tags_all`, `prohibited_tags` — and runs only when the run's tags satisfy that gate, so
switching on a whole family of scoring is a list edit rather than a code change. `ppi_refolding`
brings the target-templated AF3 refold plus the binder-RMSD metrics that read it; `apo_refolding`
adds the binder-alone fold. An entry with no gate at all runs on every design. The rule cuts both
ways: a metric whose tag you never pass is skipped silently, which is nearly always the reason an
expected column is missing from `per_example_metrics.csv`. Full semantics in the
[metric reference](../guides/metrics.md#metric-reference) and
[custom_metrics.md](../guides/custom_metrics.md#step-2--register-in-the-config-file).

**Pre-oracle transforms** decide *what* gets folded. A transform hangs off an oracle's
`pre_oracle_transform` key, rewrites the generated structure on its way to the oracle, and may fan
one design out into several outputs — each folded separately, saved under its own `save_id`, and
reported as `<oracle_key>.<save_id>.<metric>`. Two ship ready to use: `ApoOnlyTransform`, which is
what the `apo_refolding` tag runs to fold the designed chain alone, and
`OffTargetSubstitutionTransform`, which this tutorial is built on — keep the binder, swap its
partner for a structure off disk. Neither is specific to binder design; how the resulting refolds
are addressed, tagged and cached, and what else the mechanism is good for, is in
[pre-oracle transforms](../guides/metrics.md#pre-oracle-transforms).

## Setup

Paths and Slurm resources, plus the two helpers used throughout: `launch` submits a run config
and waits for its metrics table, `verdict_table` renders one row per design.

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*NVML.*")
import os
import re
import shutil
import subprocess
import time
from pathlib import Path

import pandas as pd
import yaml

# Locate the repository by walking up to the `.project-root` marker; $PROJECT_ROOT is the
# fallback when the notebook is run from outside the tree.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / ".project-root").exists() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / ".project-root").exists():
    REPO_ROOT = Path(os.environ["PROJECT_ROOT"])

# The Trk structures ship with the repository; inputs and run configs are written under OUT_ROOT.
TARGETS = REPO_ROOT / "examples/trka_crossreactivity/targets"
OUT_ROOT = Path(os.environ.get("PPI_TUTORIAL_RUNS", REPO_ROOT / "runs/ppi_specificity_tutorial"))
CFG_DIR = OUT_ROOT / "configs"
CFG_DIR.mkdir(parents=True, exist_ok=True)

# Slurm settings, all overridable by environment variable so the notebook moves between clusters.
CLUSTER = os.environ.get("CLUSTER", "digs")
PARTITION = os.environ.get("SLURM_PARTITION", "gpu")
GRES = os.environ.get("SLURM_GRES", "gpu:small:1")
MEM = os.environ.get("SLURM_MEM", "32g")
TIME = os.environ.get("SLURM_TIME", "4:00:00")
OVERWRITE = False  # False reuses existing outputs and skips sampling; True re-runs

os.environ.setdefault("PROJECT_ROOT", str(REPO_ROOT))


def launch(config: Path, out_dir: Path, njobs: int = 4, timeout: int = 14400) -> Path:
    """Submit a run config to Slurm and wait until its metrics table lands.

    The config is a complete inference experiment living outside ``configs/`` — the
    ``--config-path`` pattern documented in ``configs/inference_experiment/TEMPLATE.yaml``.
    Replicates are spread over ``njobs`` array tasks, one GPU each, and merged by the
    concatenation job the launch script queues behind them.
    """
    csv = out_dir / "inference_metrics/per_example_metrics.csv"
    if csv.exists() and not OVERWRITE:
        print(f"[skip] {config.stem}: reusing {len(pd.read_csv(csv))} designs (OVERWRITE=True re-runs)")
        return csv
    shutil.rmtree(out_dir, ignore_errors=True)
    # Create the sample directory up front: array tasks racing to mkdir it on NFS can fail.
    (out_dir / "inference_samples").mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    cmd = (f'bash "{REPO_ROOT}/scripts/launch_infer.sh" {CLUSTER} --njobs {njobs} '
           f'--partition {PARTITION} --gres {GRES} --mem {MEM} --time {TIME} '
           f'--code-dir "{REPO_ROOT}" --config-path "{config.parent}" --config-name {config.stem}')
    out = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=REPO_ROOT).stdout
    job = re.search(r"Submitted batch job (\d+)", out)
    print(f"[submit] {config.stem} over {njobs} tasks" + (f" (job {job.group(1)})" if job else ""))
    deadline = time.time() + timeout
    while time.time() < deadline:
        if csv.exists():
            print(f"[done] {config.stem}: {len(pd.read_csv(csv))} designs in {(time.time() - t0) / 60:.0f} min")
            return csv
        time.sleep(60)
    raise TimeoutError(f"{config.stem} produced no metrics before the timeout")


def _label(key: str, bound: dict) -> str:
    """Column header for one threshold, e.g. ``htrkb.min_ipae >=2.5``."""
    stem = (key.replace("oracle_ppi.", "on-target ")
               .replace("oracle_off_target.", "")
               .replace("refolded_target_aligned_binder_rmsd.rmsd", "dock rmsd")
               .replace("refolded_apo_holo_rmsd.apo.rmsd", "apo-holo rmsd")
               .replace("_coi", ""))
    op, value = ("<=", bound["max"]) if "max" in bound else (">=", bound["min"])
    return f"{stem} {op}{value:g}"


def verdict_table(csv: Path, config: Path, criterion: str):
    """One row per design, green where the config's pass criterion is met and red where it is not.

    Thresholds are read back out of the run config, so the table and the ``<criterion>.pass``
    column computed by the pipeline can never drift apart.
    """
    thresholds = yaml.safe_load(Path(config).read_text())["metrics"]["pass_criteria"][criterion]["metric_thresholds"]
    d = pd.read_csv(csv)
    ids = d["example_id"].astype(str).str.split("__").str[-1]
    if ids.duplicated().any():                       # the styled table needs a unique index
        ids = ids + "#" + ids.groupby(ids).cumcount().astype(str)

    met = pd.DataFrame({_label(k, b): (d[k] <= b["max"] if "max" in b else d[k] >= b["min"])
                        for k, b in thresholds.items()}).set_index(ids)
    show = pd.DataFrame({_label(k, b): d[k].round(2) for k, b in thresholds.items()}).set_index(ids)
    show["PASS"] = d[f"{criterion}.pass"].astype(int).to_numpy()
    order = show.sort_values(["PASS", show.columns[0]], ascending=[False, True]).index

    colours = met.map(lambda ok: "background-color:#d8f0d8" if ok else "background-color:#f7d7d7")
    colours["PASS"] = ""
    print(f"{int(show['PASS'].sum())}/{len(show)} designs pass `{criterion}`")
    return show.loc[order].style.apply(lambda _: colours.loc[order], axis=None)


print("repository :", REPO_ROOT)
print("outputs    :", OUT_ROOT)

## 1. The design job

One target, one epitope, one de novo binder — the *same* design job feeds both halves of this
tutorial. Only the scoring differs.

| Structure | Chain | Residues | Role |
|---|---|---|---|
| hTrkA | A | 278–382 | on-target in both cases; hotspots 294, 296, 333 |
| hTrkB | A | 278–382 | off-target to **avoid** (section 2) |
| hTrkC | A | 295–399 | off-target to **avoid** (section 2) |
| mTrkA | A | 282–385 | orthologue to **hit** (section 3) |
| rTrkA | D | 853–950 | orthologue to **hit** (section 3) |

The input is a [condition spec](../condition_spec_yaml.md): the target backbone is
coordinate-conditioned (`C_CRD`) so the receptor is held rigid while its side chains repack, its
sequence is revealed (`C_SEQ`), and the RFdiffusion-paper TrkA epitope is marked as hotspots
(`C_HOT`) to steer the binder onto the right face. The binder chain carries no conditions at all
— it is generated from noise, at a length the model picks between 55 and 75 residues.

In [ ]:
SPEC = CFG_DIR / "trka_binder.yaml"
SPEC.write_text(f"""
global_args:
  input_as_annotated_cif: false

trka_binder:
  input: {TARGETS}/hTrkA.cif
  contig: "A278-382, /0, BINDER:55-75"      # fixed target, then a de novo 55-75 residue binder
  conditions:
    C_CRD:                                  # hold the target backbone; side chains stay free
      true:
        - select: "sel('A') and is_protein_backbone"
    C_SEQ:                                  # reveal the target sequence
      true:
        - select: "sel('A')"
    C_HOT:                                  # steer the binder onto the epitope
      true:
        - select: "sel('A/*/[294, 296, 333]')"
""".lstrip())
print(SPEC)

The epitope, in orange, on the hTrkA fragment the binder is designed against.

In [ ]:
import io

import py3Dmol
from atomworks.io import parse
from biotite.structure.io.pdb import PDBFile


def to_pdb(atom_array) -> str:
    """Render an AtomArray as a PDB string for the viewer."""
    pdb = PDBFile()
    pdb.set_structure(atom_array)
    buf = io.StringIO()
    pdb.write(buf)
    return buf.getvalue()


target = parse(f"{TARGETS}/hTrkA.cif")["assemblies"]["1"][0]
target = target[target.mask("~has_nan_coord()")]   # unresolved side-chain atoms
view = py3Dmol.view(width=640, height=420)
view.addModel(to_pdb(target), "pdb")
view.setStyle({"cartoon": {"color": "0xb9c0c9"}})
view.addStyle({"resi": [294, 296, 333]}, {"stick": {"colorscheme": "orangeCarbon", "radius": 0.3}})
view.setBackgroundColor("0xffffff")
view.zoomTo()
view.show()

## 2. Specificity: bind hTrkA, avoid hTrkB and hTrkC

Specificity is a scoring question, so it lives in the run config rather than in the design spec.
Three blocks turn it on:

1. **`metrics_tags`** — `ppi_refolding` folds the designed complex (holo), `apo_refolding` folds
   the binder on its own, and `off_target_refolding` enables the oracle below.
2. **`oracle_off_target`** — an AF3 oracle whose `pre_oracle_transform` keeps the designed binder,
   substitutes each structure in `off_target_paths` for the target, and refolds. Every entry gets
   its own refold, reported under `oracle_off_target.<name>.*`. `template_query` hands AF3 the
   off-target's known structure, so the fold is about the interface, not the receptor.
3. **`pass_criteria`** — one criterion bundling on-target binding *and* the off-target bars. The
   direction lives entirely in the threshold: `max` on the on-target `min_ipae_coi` demands
   binding, `min` on an off-target demands the opposite.

Cost is linear in what you ask for: **one AF3 fold per design per entry**, on top of the holo and
apo folds. Two off-targets means four folds per design.

In [ ]:
SPEC_CFG = CFG_DIR / "trka_specificity.yaml"
SPEC_CFG.write_text(f"""
defaults:
  - inference
  - _self_

hydra:
  searchpath:                                # lets this out-of-tree config find configs/
    - file://${{oc.env:PROJECT_ROOT}}/configs

inputs: {SPEC}
out_dir: {OUT_ROOT}/specificity
run_name_: trka_specificity
num_replicates: 12
batch_size: 3

metrics_tags: [design, hotspot_conditioning, ppi_refolding, apo_refolding, off_target_refolding]

metrics:
  refolding_oracles:
    # Specificity assay: keep the binder, swap in each off-target, refold once apiece.
    oracle_off_target:
      _target_: rfproteina.metrics.oracles.af3_oracle.AF3Oracle
      oracle_name: af3_off_target
      template_query: "is_loaded_target & is_protein"
      aggregation: [median]
      pre_oracle_transform:
        _target_: rfproteina.metrics.oracles.pre_oracle_transform.OffTargetSubstitutionTransform
        off_target_paths:
          htrkb: {TARGETS}/hTrkB.cif
          htrkc: {TARGETS}/hTrkC.cif
      required_tags_any: [off_target_refolding]

  pass_criteria:
    trka_specific:
      _target_: rfproteina.metrics.oracles.pass_criterion_metric.PassCriterionMetric
      metric_thresholds:
        oracle_ppi.min_ipae_coi: {{max: 1.5}}                   # binds hTrkA
        oracle_ppi.iptm_coi: {{min: 80.0}}
        refolded_target_aligned_binder_rmsd.rmsd: {{max: 2.0}}  # at the epitope it was aimed at
        refolded_apo_holo_rmsd.apo.rmsd: {{max: 1.5}}           # and holds its fold apo
        oracle_off_target.htrkb.min_ipae_coi: {{min: 5.0}}      # does NOT bind hTrkB
        oracle_off_target.htrkc.min_ipae_coi: {{min: 5.0}}      # does NOT bind hTrkC
      required_tags_any: [off_target_refolding]
""".lstrip())

spec_csv = launch(SPEC_CFG, OUT_ROOT / "specificity")

### Results

`min_ipae_coi` is AF3's minimum interface PAE in Å — low means a confident interface. The
on-target columns want it low, the off-target columns want it high; each cell is green where the
config's threshold is met.

The exclusion bar sits far above the binding bar, at 5 Å rather than just outside 1.5: what we
want from an off-target is AF3 declining to model an interface at all, not modelling a slightly
worse one. A binder scoring 2.5 Å against hTrkB is not specific, it is merely less good at
hTrkB — and that gap does not survive contact with a real cell.

In [ ]:
verdict_table(spec_csv, SPEC_CFG, "trka_specific")

The two halves of the question are independent: a design can bind hTrkA beautifully and still
light up hTrkB, or be gloriously specific about a receptor it never engaged. Bundling both into
one criterion is what makes the pipeline answer them together.

## 3. Multi-specificity: bind human, mouse and rabbit TrkA

Same design job, same oracle — all that changes is which structures go into `off_target_paths`
and which way their thresholds point. Mouse and rabbit TrkA are *wanted* here, so their bars are
`max`, exactly like the on-target one, and there is no exclusion clause at all: this run asks for
cross-reactivity, not for silence elsewhere.

The orthologue bars are the on-target bars, value for value — same `min_ipae_coi`, same
`iptm_coi`. It is tempting to grade the inherited interfaces on a curve, since the model never
got to optimise against mouse or rabbit, but a binder that only half-works in the animal model
is not usable in the animal model. Loosening the bar there would not make the binder more
cross-reactive, only the table more flattering.

In [ ]:
MULTI_CFG = CFG_DIR / "trka_multispecificity.yaml"
MULTI_CFG.write_text(f"""
defaults:
  - inference
  - _self_

hydra:
  searchpath:
    - file://${{oc.env:PROJECT_ROOT}}/configs

inputs: {SPEC}
out_dir: {OUT_ROOT}/multispecificity
run_name_: trka_multispecificity
num_replicates: 12
batch_size: 3

metrics_tags: [design, hotspot_conditioning, ppi_refolding, apo_refolding, off_target_refolding]

metrics:
  refolding_oracles:
    oracle_off_target:
      _target_: rfproteina.metrics.oracles.af3_oracle.AF3Oracle
      oracle_name: af3_off_target
      template_query: "is_loaded_target & is_protein"
      aggregation: [median]
      pre_oracle_transform:
        _target_: rfproteina.metrics.oracles.pre_oracle_transform.OffTargetSubstitutionTransform
        off_target_paths:
          mtrka: {TARGETS}/mTrkA.cif                          # wanted, not avoided
          rtrka: {TARGETS}/rTrkA.cif
      required_tags_any: [off_target_refolding]

  pass_criteria:
    trka_crossreactive:
      _target_: rfproteina.metrics.oracles.pass_criterion_metric.PassCriterionMetric
      metric_thresholds:
        oracle_ppi.min_ipae_coi: {{max: 1.5}}                  # binds the receptor it was designed on
        oracle_ppi.iptm_coi: {{min: 80.0}}
        refolded_target_aligned_binder_rmsd.rmsd: {{max: 2.0}}
        oracle_off_target.mtrka.min_ipae_coi: {{max: 1.5}}     # and the mouse orthologue, at the same bar
        oracle_off_target.mtrka.iptm_coi: {{min: 80.0}}
        oracle_off_target.rtrka.min_ipae_coi: {{max: 1.5}}     # and the rabbit one, likewise
        oracle_off_target.rtrka.iptm_coi: {{min: 80.0}}
      required_tags_any: [off_target_refolding]
""".lstrip())

multi_csv = launch(MULTI_CFG, OUT_ROOT / "multispecificity")

### Results

In [ ]:
verdict_table(multi_csv, MULTI_CFG, "trka_crossreactive")

## 4. Searching for it

Both runs above sample blind and score afterwards. [Reward-guided search](../guides/search.md) scores
partial trajectories at checkpoints instead, branching from the survivors and dropping the rest,
so the compute follows the interfaces that are already forming. It is one added `search:` block;
everything from section 2 stays as it is.

The reward asks for what the criterion asks for. `reward_metrics_tags` turns on all three refolds
during scoring, so a candidate is judged on its on-target interface *and* on both off-targets:
`iptm_coi` up and `min_ipae_coi` down on hTrkA, and `min_ipae_coi` **up** on hTrkB and hTrkC — a
positive weight is how you ask for a metric to be large. The interface-PAE terms are clipped where
their pass bars sit, so a candidate already past one earns nothing more for exceeding it and the
reward's remaining range goes to whatever is still failing. `iptm_coi` is left unclipped on
purpose: it is what still separates two candidates once both are over every bar.

Because every metric the criterion reads is now computed at each checkpoint, the save criterion
can simply *be* the criterion: `trka_specific.pass` is already sitting in each candidate's metrics
dict, so thresholding on it keeps exactly the lost-beam candidates that would pass the final
table, on the same bars, with no second set of numbers to drift.

The bill: scoring one candidate is now four AF3 folds (holo, apo, two off-targets) rather than
one, at every checkpoint of every branch. If that is too much for a first pass, drop
`off_target_refolding` from `reward_metrics_tags` along with the two off-target weights — the
search then steers on the on-target interface alone at a quarter of the cost, and the save
criterion has to fall back to a bar on `search.reward`, since the off-target columns no longer
exist at checkpoint time.

In [ ]:
SEARCH_CFG = CFG_DIR / "trka_specificity_search.yaml"
SEARCH_CFG.write_text(
    SPEC_CFG.read_text()
    .replace(f"out_dir: {OUT_ROOT}/specificity", f"out_dir: {OUT_ROOT}/specificity_search")
    .replace("run_name_: trka_specificity", "run_name_: trka_specificity_search")
    .replace("num_replicates: 12\nbatch_size: 3", "num_replicates: 4\nbatch_size: 1")
    + """
generation:
  args:
    nsteps: 100                       # must equal the last step checkpoint

search:
  strategy:
    _target_: rfproteina.search.beam_search.BeamSearchStrategy
    step_checkpoints: [0, 50, 100]    # score and prune here
    n_branch: 2                       # candidates spawned per surviving beam
    beam_width: 2                     # candidates kept after each prune
    intermediate_sample_save_criterion:    # keep lost-beam candidates that already pass
      _target_: rfproteina.metrics.oracles.pass_criterion_metric.PassCriterionMetric
      metric_thresholds:
        trka_specific.pass: {min: 1.0}     # the run's own criterion, scored at the checkpoint
  rewards:
    reward_metrics_tags: [ppi_refolding, apo_refolding, off_target_refolding]
    weights:
      oracle_ppi.iptm_coi: 1.0
      oracle_ppi.min_ipae_coi: -20.0
      oracle_off_target.htrkb.min_ipae_coi: 10.0   # positive: a *failed* hTrkB interface is the goal
      oracle_off_target.htrkc.min_ipae_coi: 10.0
    clip_values:
      oracle_ppi.min_ipae_coi: {min: 0.5, max: 5.0}
      oracle_off_target.htrkb.min_ipae_coi: {max: 5.0}   # nothing more to earn past the pass bar
      oracle_off_target.htrkc.min_ipae_coi: {max: 5.0}
""")

search_csv = launch(SEARCH_CFG, OUT_ROOT / "specificity_search")

### Results

Search can return more designs than replicates: the final beam, plus every candidate that already
cleared `trka_specific` on its way out of it.

In [ ]:
verdict_table(search_csv, SEARCH_CFG, "trka_specific")

In [ ]:
for name, csv in [("unguided (section 2)", spec_csv), ("beam search", search_csv)]:
    d = pd.read_csv(csv)
    print(f"{name:22s} n={len(d):3d}   median on-target min_ipae {d['oracle_ppi.min_ipae_coi'].median():5.2f}"
          f"   best {d['oracle_ppi.min_ipae_coi'].min():5.2f}"
          f"   passing {int(d['trka_specific.pass'].sum())}")

## Summary

| To change | Edit |
|---|---|
| Which structures are assayed | `off_target_paths` — one AF3 fold per entry per design |
| Avoid vs. hit an off-target | `min` vs. `max` on its `min_ipae_coi` threshold |
| How hard the bar is | the threshold values under `metrics.pass_criteria.<name>` |
| What search optimises | `search.rewards.weights` (and `reward_metrics_tags` for other oracles) |
| What search keeps beyond the final beam | `intermediate_sample_save_criterion`; thresholding it on `<criterion>.pass` needs that criterion's metrics in `reward_metrics_tags` |
| How hard search looks | `beam_width`, `n_branch`, `step_checkpoints` |

The off-target oracle knows nothing about biology — it folds the binder against whatever you hand
it. Anti-targets, orthologues and family members are one mechanism pointed in different
directions, so a real campaign usually names all of them in a single `off_target_paths` block and
writes one criterion over the lot.

For the machinery behind these configs see the [search guide](../guides/search.md), the
[metrics guide](../guides/metrics.md), and `configs/metrics/refolding_oracles/af3.yaml`, which documents
every oracle the tags can switch on.